
# SU(3) \(O(y^4)\) cancellation-and-leakage benchmark

This notebook is a representation-neutral acceptance test for an SU(3) Hamiltonian encoding.

It tests:

1. **Cancellation:** the oriented elementary-cube boundary has no non-rigid leakage through \(O(y^3)\).
2. **Leakage:** the complete fourth-order effective Hamiltonian produces a 30-plaquette residual with maximum exact coefficient \(5/48\).

The notebook embeds the canonical 189-entry real-space \(H_4\) kernel and final certificates.

## Acceptance levels

- **Level A:** reproduce the effective \(H_4\) kernel.
- **Level B:** also reproduce vanishing cube leakage at orders 1–3.
- **Level C:** reproduce the short-time exterior probability pattern.

Required intermediate irreps:

\[
1,\;3,\;\bar3,\;6,\;\bar6,\;8,\;10,\;\overline{10},\;15,\;\overline{15}.
\]


In [1]:

from __future__ import annotations

import base64, gzip, json
from collections import defaultdict
from fractions import Fraction
from pathlib import Path
from typing import Dict, Mapping, Sequence, Tuple

KERNEL_GZ_B64 = 'H4sICAAAAAAC/3k0X2Z1bGxfcmVhbF9zcGFjZV9INF9rZXJuZWwuanNvbgDNXE1v00AQ/S8+x83MrD/W3BAqHweERHtBCEVu7JRAm1R1WgRV/zvb9IBQaxTP7jwhRT24iV9m5828l51N7rLv/fWmv8hefL7LuvVwddEu+8t+swsXcpnRjL7MsvXm6ma3CP/a9OEyzThc297snl780a/Pv4aXZq6iQtiVPC/ISV0RN048UXY/ewrDs/CIxRHiygWEpqqKURwKSIh4tMuWC5UBhFxDJdfNnKUqpA6IVO7xRuEgUSGSFDIkAM49EAHBBdJTW8OFPRwiKnUh5UwllaV33j9gMZHnhqjwIa5RLOUC1p58VXDDgWvOu4rruSNhqhvxVcN11fwDExYeiIdYGmJYiOgUIHECaRNWmjDKhBEmmBeawgPBeJS/YPLJrkEZT25nTwTkGwTjG/TLtldyAYSjZnUuztdc+EaEhWt2rplLkFQJes5lUVDjiyTe4S/USFBGZI61mZuOw/hFVNM/EhVR2wJpiZP8kcQ4CkBDBOkVSq4wajXJTXCc+h6cHgapL0eVzuHyy5GyKIhw1DTAqC+n1XwGZA5FRAQPOaKurD5Lc1R3VdeVlfQlZLi+10aCMiJzICJCeCh6gkyFIYjIj5SvGMyDWIWTxrVY4LD5uo2OTgxgANkZN3uCGdEcEE+UT1GTPBZVEMkjbe4m4xB+EbXlHGtsBVANjGiJYzstgpndGMAAyA6SK4xajR4ukdRbIFM4YLDVMgXnwEMsunCSHi5JHhUiSRMthWAshUQeIJjSfNJvVQlmUGRDb9IXU/LDJXYRItIGqitgWSGqCiROIG3CShNGmTDC9B96ITY4XCLJpzQCCGeSa2CMa0g6MDncQDBG2WNnXbpFxOwIWEyDJPUmvLqu8jRjJwsgOClYX1i5lbpz+hMYYj7sQsSG0EWJaPBGbiJG5Z+H4eTTE3mWAmwydULEM9IXDAJ6ti8waIzGtpvwIztTbLwLv4cVRPJQZERwkfS1NRkGwHjSV1a8lWXAnInR1UzqqkpgoNlghgZgob7txk+DbFT+4GhiTAsrYXJDNyEgkU+9pXPQusFEXqLUVrmKuZ2sSzqrxDNMdIIoLtLXlqHMp96yMiYFqSsrxdeHGDIZsm5PpK6r+O/xsMGMCFBT+gZoJr8Mkl/GyC9HTj6naC5Haq59hkZmRAyaEU3DOejnUbSrlvD3RNJH9ezhEsaMoOwWT12ziQ+XWEaISBuotLCVhSksRPMDiS1Ia4FSi1FaC6ENz7rsd2324i47a4f18HiHIdzicRfo0bzt7xeeOuza896tF/t3EO47v9zs5l27a+efisXrjx/eL05efXx5+urt4vT45PTh4snpyzfH7t38Z7FYbi+vLvpdv1htL7q+W/zYXoc/+7c4HH0btpuj81/ZH4zhaytlFUB8WbQkfeM5WPiur1fLVcctF11XLptQne1qRUtXtdJ67tuQtqbrVsVq5ZdO+KyjcM/b/npYbzcPH7RIqpyqnF3+CPQtv+Xs/v43CpkthD9PAAA='
RESIDUAL_JSON = '{"H4_cube_image":[{"coefficient":"17607806155349/2202655210329600","plaquette":[-1,0,0,0,1]},{"coefficient":"-17607806155349/2202655210329600","plaquette":[-1,0,0,0,2]},{"coefficient":"5/48","plaquette":[-1,0,0,1,2]},{"coefficient":"-17607806155349/2202655210329600","plaquette":[-1,0,1,0,1]},{"coefficient":"17607806155349/2202655210329600","plaquette":[-1,1,0,0,2]},{"coefficient":"17607806155349/2202655210329600","plaquette":[0,-1,0,0,1]},{"coefficient":"-5/48","plaquette":[0,-1,0,0,2]},{"coefficient":"17607806155349/2202655210329600","plaquette":[0,-1,0,1,2]},{"coefficient":"-17607806155349/2202655210329600","plaquette":[0,-1,1,0,1]},{"coefficient":"5/48","plaquette":[0,0,-1,0,1]},{"coefficient":"-17607806155349/2202655210329600","plaquette":[0,0,-1,0,2]},{"coefficient":"17607806155349/2202655210329600","plaquette":[0,0,-1,1,2]},{"coefficient":"4555981615057344457/1812647572150615200","plaquette":[0,0,0,0,1]},{"coefficient":"-4555981615057344457/1812647572150615200","plaquette":[0,0,0,0,2]},{"coefficient":"4555981615057344457/1812647572150615200","plaquette":[0,0,0,1,2]},{"coefficient":"-4555981615057344457/1812647572150615200","plaquette":[0,0,1,0,1]},{"coefficient":"-17607806155349/2202655210329600","plaquette":[0,0,1,0,2]},{"coefficient":"17607806155349/2202655210329600","plaquette":[0,0,1,1,2]},{"coefficient":"-5/48","plaquette":[0,0,2,0,1]},{"coefficient":"17607806155349/2202655210329600","plaquette":[0,1,-1,0,2]},{"coefficient":"17607806155349/2202655210329600","plaquette":[0,1,0,0,1]},{"coefficient":"4555981615057344457/1812647572150615200","plaquette":[0,1,0,0,2]},{"coefficient":"17607806155349/2202655210329600","plaquette":[0,1,0,1,2]},{"coefficient":"-17607806155349/2202655210329600","plaquette":[0,1,1,0,1]},{"coefficient":"17607806155349/2202655210329600","plaquette":[0,1,1,0,2]},{"coefficient":"5/48","plaquette":[0,2,0,0,2]},{"coefficient":"-17607806155349/2202655210329600","plaquette":[1,-1,0,1,2]},{"coefficient":"-17607806155349/2202655210329600","plaquette":[1,0,-1,1,2]},{"coefficient":"17607806155349/2202655210329600","plaquette":[1,0,0,0,1]},{"coefficient":"-17607806155349/2202655210329600","plaquette":[1,0,0,0,2]},{"coefficient":"-4555981615057344457/1812647572150615200","plaquette":[1,0,0,1,2]},{"coefficient":"-17607806155349/2202655210329600","plaquette":[1,0,1,0,1]},{"coefficient":"-17607806155349/2202655210329600","plaquette":[1,0,1,1,2]},{"coefficient":"17607806155349/2202655210329600","plaquette":[1,1,0,0,2]},{"coefficient":"-17607806155349/2202655210329600","plaquette":[1,1,0,1,2]},{"coefficient":"-5/48","plaquette":[2,0,0,1,2]}],"cube_state":[{"coefficient":"-1","plaquette":[0,0,0,0,1]},{"coefficient":"1","plaquette":[0,0,0,0,2]},{"coefficient":"-1","plaquette":[0,0,0,1,2]},{"coefficient":"1","plaquette":[0,0,1,0,1]},{"coefficient":"-1","plaquette":[0,1,0,0,2]},{"coefficient":"1","plaquette":[1,0,0,1,2]}],"dominant_leakage":[{"coefficient":"5/48","plaquette":[-1,0,0,1,2]},{"coefficient":"-5/48","plaquette":[0,-1,0,0,2]},{"coefficient":"5/48","plaquette":[0,0,-1,0,1]},{"coefficient":"-5/48","plaquette":[0,0,2,0,1]},{"coefficient":"5/48","plaquette":[0,2,0,0,2]},{"coefficient":"-5/48","plaquette":[2,0,0,1,2]}],"meta":{"kernel_file":"DATA_Y4_full_real_space_h4_kernel.json.gz","kernel_sha256":"635d40fa8a5d7da841fd30f36185eb96f14ec4c040678ddd8fb010379afb2900","version":"2026-06-13-stage3j-v1"},"residual":[{"coefficient":"17607806155349/2202655210329600","plaquette":[-1,0,0,0,1]},{"coefficient":"-17607806155349/2202655210329600","plaquette":[-1,0,0,0,2]},{"coefficient":"5/48","plaquette":[-1,0,0,1,2]},{"coefficient":"-17607806155349/2202655210329600","plaquette":[-1,0,1,0,1]},{"coefficient":"17607806155349/2202655210329600","plaquette":[-1,1,0,0,2]},{"coefficient":"17607806155349/2202655210329600","plaquette":[0,-1,0,0,1]},{"coefficient":"-5/48","plaquette":[0,-1,0,0,2]},{"coefficient":"17607806155349/2202655210329600","plaquette":[0,-1,0,1,2]},{"coefficient":"-17607806155349/2202655210329600","plaquette":[0,-1,1,0,1]},{"coefficient":"5/48","plaquette":[0,0,-1,0,1]},{"coefficient":"-17607806155349/2202655210329600","plaquette":[0,0,-1,0,2]},{"coefficient":"17607806155349/2202655210329600","plaquette":[0,0,-1,1,2]},{"coefficient":"-17607806155349/2202655210329600","plaquette":[0,0,1,0,2]},{"coefficient":"17607806155349/2202655210329600","plaquette":[0,0,1,1,2]},{"coefficient":"-5/48","plaquette":[0,0,2,0,1]},{"coefficient":"17607806155349/2202655210329600","plaquette":[0,1,-1,0,2]},{"coefficient":"17607806155349/2202655210329600","plaquette":[0,1,0,0,1]},{"coefficient":"17607806155349/2202655210329600","plaquette":[0,1,0,1,2]},{"coefficient":"-17607806155349/2202655210329600","plaquette":[0,1,1,0,1]},{"coefficient":"17607806155349/2202655210329600","plaquette":[0,1,1,0,2]},{"coefficient":"5/48","plaquette":[0,2,0,0,2]},{"coefficient":"-17607806155349/2202655210329600","plaquette":[1,-1,0,1,2]},{"coefficient":"-17607806155349/2202655210329600","plaquette":[1,0,-1,1,2]},{"coefficient":"17607806155349/2202655210329600","plaquette":[1,0,0,0,1]},{"coefficient":"-17607806155349/2202655210329600","plaquette":[1,0,0,0,2]},{"coefficient":"-17607806155349/2202655210329600","plaquette":[1,0,1,0,1]},{"coefficient":"-17607806155349/2202655210329600","plaquette":[1,0,1,1,2]},{"coefficient":"17607806155349/2202655210329600","plaquette":[1,1,0,0,2]},{"coefficient":"-17607806155349/2202655210329600","plaquette":[1,1,0,1,2]},{"coefficient":"-5/48","plaquette":[2,0,0,1,2]}],"rigid_component_c4":"-4555981615057344457/1812647572150615200"}'
VERDICT_JSON = '{"files":{"y4_cube_boundary_residual.json":{"residual_records":30,"sha256":"54f287d3bca14393b17ef5c63a358e87e4a0e8acffeee97b33e2c476183829ba"},"DATA_Y4_full_real_space_h4_kernel.json.gz":{"records":189,"sha256":"635d40fa8a5d7da841fd30f36185eb96f14ec4c040678ddd8fb010379afb2900"}},"gates":{"J0_real_space_kernel":{"exact_hermiticity":true,"nonzero_full_kernel_entries":189,"nonzero_root_kernel_entries":63,"ordered_words":4221,"passed":true,"proper_cubic_covariance":true,"stage3i_sha256":"854a02e981098de7fcfd1a14dd5c9703aff0c36a2a81ea5589ddf4ff8c321bd0"},"J1_cube_boundary":{"H4_cube_support":36,"cube_faces":6,"dominant_leakage_count":6,"flatness_through_O_y4":false,"maximum_residual_magnitude":"5/48","nonzero_residual_plaquettes":30,"passed":true,"rigid_component_c4":"-4555981615057344457/1812647572150615200"},"J2_dispersion_witness":{"decimal_lower_bound_on_bandwidth_coefficient":0.06395120243159332,"nonzero_dispersion":true,"one_pi_correction":"-17700498622147435111/7250590288602460800","passed":true,"three_pi_correction":"-3447362930970494909/1450118057720492160","three_pi_minus_one_pi":"17607806155349/275331901291200","two_pi_correction":"-4367164159624988707/1812647572150615200"},"J3_round_trip":{"kernel_records":189,"passed":true,"residual_records":30}},"high_symmetry_corrections":{"one_pi_x":"-17700498622147435111/7250590288602460800","one_pi_y":"-17700498622147435111/7250590288602460800","one_pi_z":"-17700498622147435111/7250590288602460800","three_pi":"-3447362930970494909/1450118057720492160","two_pi_xy":"-4367164159624988707/1812647572150615200","two_pi_xz":"-4367164159624988707/1812647572150615200","two_pi_yz":"-4367164159624988707/1812647572150615200"},"meta":{"a100_required":false,"date":"2026-06-13","hardware":"CPU","platform":"Linux-4.4.0-x86_64-with-glibc2.41","python":"3.13.5 (main, Jun 25 2025, 18:55:22) [GCC 14.2.0]","version":"2026-06-13-stage3j-v1","walltime_s":0.21193170547485352},"passed":true,"scope":{"claim":"strong-coupling one-flux effective Hamiltonian through O(y^4)","not_claimed":["all-orders behavior","continuum glueball bandwidth","continuum Yang-Mills mass gap"]},"verdict":{"cube_boundary_residual_nonzero":true,"dispersion_witness_decimal":0.06395120243159332,"exact_dispersion_witness":"17607806155349/275331901291200","first_nonzero_bandwidth_order":"y^4","flat_through_order_y4":false,"maximum_exact_leakage":"5/48","rigid_component_c4":"-4555981615057344457/1812647572150615200","statement":"The T1^{+-} one-flux band is exactly flat through O(y^3) but acquires nonzero dispersion at O(y^4)."}}'

kernel_payload = json.loads(
    gzip.decompress(base64.b64decode(KERNEL_GZ_B64)).decode("utf-8")
)
residual_certificate = json.loads(RESIDUAL_JSON)
verdict_certificate = json.loads(VERDICT_JSON)

PLANES = ((0, 1), (0, 2), (1, 2))
PLANE_INDEX = {plane: i for i, plane in enumerate(PLANES)}

def Q(x):
    if isinstance(x, Fraction):
        return x
    if isinstance(x, int):
        return Fraction(x, 1)
    if isinstance(x, float):
        return Fraction(str(x))
    return Fraction(str(x))

reference_kernel = {
    (
        tuple(row["input_plane"]),
        tuple(row["output_plane"]),
        tuple(row["displacement"]),
    ): Q(row["weight"])
    for row in kernel_payload["kernel"]
}

reference_cube = {
    tuple(row["plaquette"]): Q(row["coefficient"])
    for row in residual_certificate["cube_state"]
}
reference_residual = {
    tuple(row["plaquette"]): Q(row["coefficient"])
    for row in residual_certificate["residual"]
}

EXPECTED = {
    "kernel_entries": 189,
    "rigid_component_c4": Q(residual_certificate["rigid_component_c4"]),
    "residual_support": 30,
    "max_leakage": Q("5/48"),
    "dispersion_witness": Q(
        verdict_certificate["verdict"]["exact_dispersion_witness"]
    ),
}
print("Loaded exact benchmark certificate.")
print(EXPECTED)


Loaded exact benchmark certificate.
{'kernel_entries': 189, 'rigid_component_c4': Fraction(-4555981615057344457, 1812647572150615200), 'residual_support': 30, 'max_leakage': Fraction(5, 48), 'dispersion_witness': Fraction(17607806155349, 275331901291200)}


In [2]:

Vec3 = Tuple[int, int, int]
Plane = Tuple[int, int]
Plaquette = Tuple[int, int, int, int, int]
KernelKey = Tuple[Plane, Plane, Vec3]

def add3(a: Vec3, b: Vec3) -> Vec3:
    return tuple(a[i] + b[i] for i in range(3))

def apply_kernel(kernel, state):
    by_plane = defaultdict(list)
    for (pin, pout, disp), value in kernel.items():
        by_plane[pin].append((pout, disp, Q(value)))
    out = defaultdict(Fraction)
    for plaquette, coefficient in state.items():
        anchor = plaquette[:3]
        pin = plaquette[3:]
        for pout, disp, value in by_plane[pin]:
            out[add3(anchor, disp) + pout] += Q(coefficient) * value
    return {p: v for p, v in out.items() if v}

def hermiticity_defects(kernel):
    defects = {}
    for (pin, pout, d), value in kernel.items():
        reverse = (pout, pin, tuple(-x for x in d))
        defect = Q(value) - Q(kernel.get(reverse, 0))
        if defect:
            defects[(pin, pout, d)] = defect
    return defects

def cube_residual(kernel):
    image = apply_kernel(kernel, reference_cube)
    numerator = sum(
        coefficient * image.get(p, Fraction(0))
        for p, coefficient in reference_cube.items()
    )
    denominator = sum(c*c for c in reference_cube.values())
    rigid = numerator / denominator
    support = set(image) | set(reference_cube)
    residual = {
        p: image.get(p, Fraction(0)) - rigid * reference_cube.get(p, Fraction(0))
        for p in support
    }
    return image, rigid, {p: v for p, v in residual.items() if v}

def symbol_at_parity(kernel, phases):
    matrix = [[Fraction(0) for _ in range(3)] for _ in range(3)]
    for (pin, pout, disp), value in kernel.items():
        phase = 1
        for axis in range(3):
            if disp[axis] % 2:
                phase *= phases[axis]
        matrix[PLANE_INDEX[pout]][PLANE_INDEX[pin]] += phase * Q(value)
    return matrix

def flat_vector(phases):
    return (phases[2] - 1, -(phases[1] - 1), phases[0] - 1)

def rayleigh(vector, matrix):
    norm = sum(x*x for x in vector)
    return sum(
        vector[i] * matrix[i][j] * vector[j]
        for i in range(3) for j in range(3)
    ) / norm

def dispersion_witness(kernel):
    one = rayleigh(flat_vector((-1,1,1)), symbol_at_parity(kernel, (-1,1,1)))
    three = rayleigh(flat_vector((-1,-1,-1)), symbol_at_parity(kernel, (-1,-1,-1)))
    return three - one

def compare_sparse_maps(actual, expected):
    keys = set(actual) | set(expected)
    return {
        k: Q(actual.get(k,0)) - Q(expected.get(k,0))
        for k in keys
        if Q(actual.get(k,0)) != Q(expected.get(k,0))
    }

def validate_h4_kernel(kernel, exact=True, atol=1e-12):
    defects = hermiticity_defects(kernel)
    image, rigid, residual = cube_residual(kernel)
    witness = dispersion_witness(kernel)
    max_leakage = max((abs(v) for v in residual.values()), default=Fraction(0))
    residual_difference = compare_sparse_maps(residual, reference_residual)

    if exact:
        checks = {
            "kernel_entry_count": len(kernel) == EXPECTED["kernel_entries"],
            "hermitian": not defects,
            "rigid_component": rigid == EXPECTED["rigid_component_c4"],
            "residual_support": len(residual) == EXPECTED["residual_support"],
            "max_leakage": max_leakage == EXPECTED["max_leakage"],
            "residual_entrywise": not residual_difference,
            "dispersion_witness": witness == EXPECTED["dispersion_witness"],
        }
    else:
        checks = {
            "kernel_entry_count": len(kernel) == EXPECTED["kernel_entries"],
            "hermitian": max((abs(float(v)) for v in defects.values()), default=0.0) <= atol,
            "rigid_component": abs(float(rigid - EXPECTED["rigid_component_c4"])) <= atol,
            "residual_support": len(residual) == EXPECTED["residual_support"],
            "max_leakage": abs(float(max_leakage - EXPECTED["max_leakage"])) <= atol,
            "residual_entrywise": max(
                (abs(float(v)) for v in residual_difference.values()),
                default=0.0,
            ) <= atol,
            "dispersion_witness": abs(
                float(witness - EXPECTED["dispersion_witness"])
            ) <= atol,
        }

    return {
        "passed": all(checks.values()),
        "checks": checks,
        "observed": {
            "kernel_entries": len(kernel),
            "rigid_component_c4": str(rigid),
            "residual_support": len(residual),
            "max_leakage": str(max_leakage),
            "dispersion_witness": str(witness),
            "hermiticity_defects": len(defects),
            "residual_mismatches": len(residual_difference),
        },
    }

def validate_lower_order_cube_actions(actions, exact=True, atol=1e-12):
    results = {}
    for order in (1,2,3):
        action = actions.get(order, {})
        numerator = sum(
            reference_cube[p] * Q(action.get(p,0))
            for p in reference_cube
        )
        denominator = sum(c*c for c in reference_cube.values())
        rigid = numerator / denominator
        residual = {
            p: Q(action.get(p,0)) - rigid * reference_cube.get(p,0)
            for p in set(action) | set(reference_cube)
        }
        residual = {p:v for p,v in residual.items() if v}
        leakage = max((abs(v) for v in residual.values()), default=Fraction(0))
        passed = leakage == 0 if exact else float(leakage) <= atol
        results[order] = {
            "passed": passed,
            "rigid_shift": str(rigid),
            "residual_support": len(residual),
            "max_leakage": str(leakage),
        }
    return {"passed": all(x["passed"] for x in results.values()), "orders": results}

REQUIRED_IRREPS = {
    (0,0): "1", (1,0): "3", (0,1): "3bar", (2,0): "6",
    (0,2): "6bar", (1,1): "8", (3,0): "10", (0,3): "10bar",
    (2,1): "15", (1,2): "15bar",
}

def validate_irrep_coverage(available):
    available = {tuple(x) for x in available}
    missing = {
        ir: name for ir, name in REQUIRED_IRREPS.items()
        if ir not in available
    }
    return {
        "passed": not missing,
        "available_count": len(available),
        "required_count": len(REQUIRED_IRREPS),
        "missing": {str(k): v for k,v in missing.items()},
    }

def load_candidate_json(path):
    payload = json.loads(Path(path).read_text())
    kernel = {
        (
            tuple(row["input_plane"]),
            tuple(row["output_plane"]),
            tuple(row["displacement"]),
        ): Q(row["weight"])
        for row in payload.get("h4_kernel", [])
    }
    actions = {}
    for order, rows in payload.get("lower_order_cube_actions", {}).items():
        actions[int(order)] = {
            tuple(row["plaquette"]): Q(row["coefficient"])
            for row in rows
        }
    return payload, kernel, actions


## Reference-certificate self-test

In [3]:

reference_report = validate_h4_kernel(reference_kernel, exact=True)
assert reference_report["passed"], reference_report
print(json.dumps(reference_report, indent=2))


{
  "passed": true,
  "checks": {
    "kernel_entry_count": true,
    "hermitian": true,
    "rigid_component": true,
    "residual_support": true,
    "max_leakage": true,
    "residual_entrywise": true,
    "dispersion_witness": true
  },
  "observed": {
    "kernel_entries": 189,
    "rigid_component_c4": "-4555981615057344457/1812647572150615200",
    "residual_support": 30,
    "max_leakage": "5/48",
    "dispersion_witness": "17607806155349/275331901291200",
    "hermiticity_defects": 0,
    "residual_mismatches": 0
  }
}



## Candidate encoding adapter

Export JSON following `candidate_schema.json`.

At minimum, provide:

- `available_irreps_dynkin`;
- the translation-invariant one-flux `h4_kernel`.

For a microscopic benchmark, also provide `lower_order_cube_actions` for orders 1–3. Each action may contain a rigid shift on the cube state, but its orthogonal residual must vanish.


In [4]:

CANDIDATE_JSON = None  # e.g. "/content/my_encoding_export.json"

if CANDIDATE_JSON is None:
    print("Candidate mode is ready. Set CANDIDATE_JSON to run it.")
else:
    payload, candidate_kernel, candidate_actions = load_candidate_json(CANDIDATE_JSON)
    report = {
        "encoding_name": payload.get("encoding_name", "unnamed"),
        "irrep_coverage": validate_irrep_coverage(
            payload.get("available_irreps_dynkin", [])
        ),
        "h4_kernel": validate_h4_kernel(
            candidate_kernel, exact=False, atol=1e-10
        ),
    }
    if candidate_actions:
        report["orders_1_to_3"] = validate_lower_order_cube_actions(
            candidate_actions, exact=False, atol=1e-10
        )
    print(json.dumps(report, indent=2))


Candidate mode is ready. Set CANDIDATE_JSON to run it.



## Short-time dynamical signature

For an exterior plaquette with residual coefficient \(r_q\),

\[
P_q(t)=y^8t^2|r_q|^2+\cdots.
\]

At each of the six dominant leakage locations,

\[
|r_q|=\frac5{48},
\qquad
P_q(t)=\frac{25}{2304}y^8t^2+\cdots.
\]


In [5]:

dominant = [
    row for row in residual_certificate["dominant_leakage"]
    if abs(Q(row["coefficient"])) == Q("5/48")
]
assert len(dominant) == 6
print("Dominant leakage sites:", len(dominant))
print("Leading probability coefficient |r_q|^2 =", Q("25/2304"))
for row in dominant:
    print(row)


Dominant leakage sites: 6
Leading probability coefficient |r_q|^2 = 25/2304
{'coefficient': '5/48', 'plaquette': [-1, 0, 0, 1, 2]}
{'coefficient': '-5/48', 'plaquette': [0, -1, 0, 0, 2]}
{'coefficient': '5/48', 'plaquette': [0, 0, -1, 0, 1]}
{'coefficient': '-5/48', 'plaquette': [0, 0, 2, 0, 1]}
{'coefficient': '5/48', 'plaquette': [0, 2, 0, 0, 2]}
{'coefficient': '-5/48', 'plaquette': [2, 0, 0, 1, 2]}


In [6]:

benchmark_self_test = {
    "reference_kernel": reference_report,
    "required_irreps": {str(k): v for k,v in REQUIRED_IRREPS.items()},
    "dominant_short_time_probability_coefficient": "25/2304",
}
output_root = Path("/content") if Path("/content").exists() else Path.cwd()
out = output_root / "CERT_Y4_su3_benchmark_self_test.json"
out.write_text(json.dumps(benchmark_self_test, indent=2))
print("Wrote", out)


Wrote /content/CERT_Y4_su3_benchmark_self_test.json
